<div style="text-align: center;">
    <h1><strong>Detección de <i>Clickbait</i> en noticias</strong></h1>
    <h2>Proyecto Procesado de Lenguaje Natural</h2>
    <br>
    <h3>Máster en Ciencia de Datos</h3>
    <h4>Universitat de València</h4>
    <br>
    <p>Juan Alcaráz Otón, Xueyao An, Fabián Calvo Castillo, Adrián Carrasco Alcalá, Javier Herrero Pérez, Mario Martínez Guillén y Clara Montalvá Barcenilla</p>
    <p><b>Curso 2025/2026</p>
</div>
<hr>

## 1. Introducción

El periodismo digital ha transformado profundamente la forma en que los usuarios consumen información, generando un ecosistema altamente competitivo donde la atención del lector es el principal activo. 
En este contexto, ha proliferado el uso del clickbait, definido como una estrategia utilizada en los medios digitales que busca llamar la atención a través de los titulares, apelando a las emociones y a la curiosidad de los lectores para forzar el click en la noticia, a menudo en perjuicio de la calidad informativa (Bravo Araujo, Serrano-Puche y Novoa Jaso, 2021).

De acuerdo con Bazaco et al. (2019), se trata de un fenómeno comunicativo dinámico que prioriza la interrogante sobre la información, omitiendo datos clave para generar un vacío de curiosidad o empleando un enfoque sensacionalista. Dentro del marco de este proyecto, se adoptará una aproximación dual para la identificación de clickbait:

* Por un lado, como un titular sensacionalista diseñado con estructuras lingüísticas concretas para captar la atención e incentivar el click, el cual puede ser detectado analizando únicamente el titular.

* Por otro lado, como un titular llamativo que presenta una diferencia considerable con el contenido de la noticia, ya sea porque plantea una pregunta que nunca se responde o porque presenta información completamnete opuesta a la que se desarrolla en el cuerpo del texto.

Para automatizar la detección de estos patrones, el proyecto emplea el modelo taniwasl/clickbait_es, alojado en Hugging Face. Este clasificador se fundamenta en BETO, la versión entrenada para el idioma español del modelo de lenguaje BERT (Bidirectional Encoder Representations from Transformers). Este modelo ha sido ajustado (fine-tuned) específicamente con un corpus aproximado de 30.000 noticias provenientes de múltiples medios españoles. 

Su funcionamiento radica en procesar la secuencia de palabras del titular analizando el contexto bidireccional de cada término mediante mecanismos de atención. Al haber aprendido de decenas de miles de ejemplos, el modelo es capaz de ponderar características semánticas y sintácticas intrínsecas del clickbait, permitiendo predecir y clasificar nuevos titulares con precisión.


## 2. Objetivo

Objetivo General:

* Analizar y clasificar el uso de técnicas de clickbait en la prensa digital española mediante la extracción automatizada de noticias y la aplicación de técnicas de Procesamiento de Lenguaje Natural.

Objetivos Específicos:

1. Implementar técnicas de web scraping para adquirir de forma automatizada noticias de las categorías Internacional, Nacional y Cultura procedentes de una selección de los principales periódicos y medios de comunicación españoles.

2. Analizar los titulares y los contenidos textuales de las noticias desde la perspectiva del NLP para extraer características distintivas que diferencien las noticias con y sin clickbait, evaluando las tendencias en función del medio y la categoría. Adicionalmente, realizar un análisis similar con los títulos y contenidos de vídeos de YouTube sobre las temáticas 11s, cambio climático y Covid clasificados como Fake New y No Fake New. 

3. Integrar un agente de Inteligencia Artificial que automatice la inferencia y el etiquetado masivo de los titulares adquiridos, clasificándolos en clickbait o no clickbait y con la capacidad añadida de generar un titular que adecuado a las noticias clasificadas como clickbait.


## 3. Metodología

### 3.1. Adquisición de los datos

Obtenemos noticias de los siguientes periódicos y medios españoles:

- ABC
- elDiario
- El Confidencial
- La Vanguardia
- 20minutos
- OkDiario
- RTVE
- Mediterráneo Digital
- El HuffPost

En particular, los periódicos digitales _Mediterráneo Digital_ y _El Huffpost_ destacan por su gran cantidad de titulares sensacionalistas y que no se corresponden con el contenido de los cuerpos de las noticias.

Extraemos los artículos de las siguientes categorías:

- Internacional: Noticias de ámbito mundial
- Nacional: Noticias de España y de sus regiones
- Cultura: Artes, cine, literatura, entretenimiento, etc.

Para ello, utilizamos tanto las páginas feed de aquellos periódicos que disponen de ellas como técnicas de scraping directo sobre los HTML de las páginas web.

Las noticias extraídas se almacenan en formato JSON con la siguiente estructura:

```json
{
  "Link": "string",
  "Periódico": "string",
  "Fecha": "string (YYYY-MM-DD)",
  "Título": "string",
  "Subtítulo": "string o null",
  "Categoría": "string",
  "Contenido": "string"
}
```

Los campos de esta estructura representan lo siguiente:

| Campo | Tipo | Descripción |
|-------|------|-------------|
| Link | string | URL del artículo original |
| Periódico | string | Nombre del medio de comunicación |
| Fecha | string | Fecha de publicación (formato YYYY-MM-DD) |
| Título | string | Título principal del artículo |
| Subtítulo | string o null | Subtítulo o descripción breve |
| Categoría | string | Categoría o sección del artículo |
| Contenido | string | Texto completo del artículo |

Se crea un JSON para cada periódico, en el que se recogen todas sus noticias, y estos ficheros se guardan en la carpeta 'data/' con el formato de nombre 'nombredelmedio.json'.

### 3.2. Preprocesado del texto

Una vez obtenidas suficientes noticias de los diferentes medios, las guardamos todas en el archivo JSON común 'conjunto_noticias.json'.

Tras una vista previa del contenido almacenado en dicho archivo, se determina que los elementos del texto que deberían eliminarse son los siguientes:

- **Comillas:** hay muchas formas diferentes que cada periódico utiliza para poner comillas, estas son: /""/, ««, '', "" y algunas más.  
- **Signos de interrogación y exclamación:** ¡! ¿?
- **Guiones:** --
- **URLs**
- **Direcciones de correo y cuentas de twitter:** empiezan por @
- **Saltos de línea:** \n \r
- **Signos de puntuación:** , ; : .
- **Corchetes y paréntesis:** [] ()
- **Emojis**

### 3.3. Clasificación de las noticias en Clickbait/No Clickbait

Una vez finalizado el preprocesado del texto y consolidado el conjunto de datos limpio en 'conjunto_noticias.json', se procede a la etapa de clasificación automática de los titulares. Para esta tarea central, se implementa el modelo preentrenado taniwasl/clickbait_es, disponible en el repositorio de Hugging Face.

El procedimiento técnico de clasificación consta de los siguientes pasos:

1. Carga del entorno de inferencia: A través de la librería transformers, se instancia tanto el tokenizador correspondiente a la arquitectura BETO como el propio modelo de clasificación de secuencias (AutoModelForSequenceClassification).

2. Tokenización de titulares: Cada valor del campo "Título" de nuestro JSON se pasa por el tokenizador. Este paso convierte el texto crudo en tensores y añade los tokens especiales de inicio y fin de oración que requiere la red neuronal para acotar el contexto.

3. Inferencia del modelo: Los tensores se introducen en el modelo preentrenado. Gracias a su fine-tuning previo sobre noticias de la prensa española, el modelo hace pasar las representaciones vectoriales por sus distintas capas ocultas. El sistema detecta patrones típicos del sensacionalismo en español como la hiperbolización, deícticos temporales o preguntas abiertas.

4. Etiquetado: La red neuronal devuelve unos valores que determinan la probabilidad de que el texto pertenezca a la clase clickbait o a la clase no clickbait. A cada noticia en el conjunto de datos se le adjunta esta nueva etiqueta predictiva.

La automatización de este proceso nos permite generar la variable objetivo para todo el conjunto de noticias. Con los datos ya etiquetados, es posible realizar un análisis cruzado para validar estadísticamente qué medios y categorías temáticas incurren con mayor frecuencia en la desinformación o en tácticas de clickbait.


### 3.4. Extracción de características

#### 3.4.1. Enfoque inicial: características lingüisticas


Se inició el análisis calculando la distancia coseno entre los embeddings del título y los fragmentos del cuerpo de cada noticia. Con este enfoque se buscaba capturar la relación semántica entre el titular y el contenido, bajo la hipótesis de que los artículos con clickbait presentarían mayores discrepancias semánticas entre ambos elementos.

Se exploraron variables lingüísticas más sofisticadas, incluyendo deícticos, intensificadores, superlativos, palabras emocionales y frases de curiosidad. Estas características fueron
seleccionadas por su relevancia teórica en la detección de clickbait, al ser indicadores habituales en titulares sensacionalistas.

#### 3.4.2. Exploración de características alternativas


Se evaluaron diferentes enfoques con el objetivo de mejorar la capacidad discriminativa del modelo: Análisis de entropía, se calculó la entropía como medida de redundancia léxica, buscando patrones en la repetición de palabras entre títulos y contenido; Análisis de sentimientos, se aplicaron métodos de análisis de sentimiento basados en BERT, bajo la hipótesis de que el clickbait podría presentar patrones emocionales diferenciados; y otras métricas, se exploraron índices de legibilidad (Flesch-Kincaid), longitud de textos y métodos de pregunta-respuesta.

Ninguna de estas características logró producir diferencias estadísticas significativas entre los grupos, lo que motivó la transición hacia un enfoque basado en representaciones neuronales.

### 3.5. Fine-tuning con BERT


Ante los resultados limitados del enfoque basado en características manuales, se recurrió al fine-tuning de un modelo de transformers. Se utilizó **bert-base-multilingual-cased** como modelo base, optimizado para el procesamiento de texto en múltiples idiomas, incluyendo el español. Con esta perspectiva, se aprenden las representaciones contextuales de los textos directamente de los datos, capturando patrones semánticos que son difíciles de formalizar utilizando métricas.

La configuración del entrenamiento fue la siguiente: Número de épocas $3$-$4$ según el modelo; learning rate de $2 \cdot 10^{-5}$, optimizador Adam, división de datos $80 \%$ entrenamiento y $20 \%$ validación; tamaño de batch de $16$ para títulos y $8$ para contenido; corrección de desbalance mediante pesos de clase inversamente proporcionales a la frecuencia; y un dropout de $0.2$ para evitar sobreajuste.

Se entrenó el modelo durante tan pocas épocas para evitar modificar demasiado los pesos del modelo preentrenado, dado el tamaño limitado del corpus. El objetivo era que el modelo aprendiera patrones específicos de clickbait sin perder la estructura semántica general de las representaciones de BERT.

Se realizó el fine-tuning de forma independiente para títulos y contenido, con longitudes máximas de secuencia de $128$ y $512$ tokens respectivamente. Los embeddings de clasificación fueron extraídos del token *[CLS]*.

BERT fue preentrenado sobre texto crudo y completo, incluyendo stopwords y formas flexionadas de las palabras, por lo que su arquitectura de atención está optimizada para procesar este tipo de entrada. Lematizar o eliminar palabras altera la estructura sintáctica y semántica que BERT ha aprendido a interpretar. Palabras como "corriendo" y "correr" tienen representaciones internas distintas que el modelo utiliza intencionalmente para capturar matices de significado. Además, los stopwords no son simplemente ruido palabras como "pero", "aunque" o "sin embargo" pueden ser indicadores lingüísticos importantes de estructura argumentativa y emotional framing, rasgos relevantes para la detección de clickbait. Es por esta razón que se ha decidido no lematizar ni eliminar las stopwords.

### 3.6 Modelo base para vídeos de YouTube

Como parte del análisis adicional realizado sobre el conjunto de datos de vídeos de YouTube, se optó por utilizar DistilBERT base uncased (distilbert-base-uncased) como modelo de extracción de representaciones semánticas. Este modelo es una versión destilada de BERT optimizada para textos en inglés, idioma en el que se encuentran los vídeos scrapeados, lo que lo hace más adecuado que bert-base-multilingual-cased para este caso concreto.

A diferencia del análisis principal realizado con las noticias en español, en este caso no fue posible llevar a cabo un proceso de fine-tuning del modelo. La razón principal es el tamaño del corpus disponible, ya que el conjunto de vídeos de YouTube cuenta únicamente con $147$ registros, distribuidos en tres temáticas (11s, Cambio climático y Covid), frente a los $540$ ejemplos utilizados en el análisis principal. Con un volumen tan reducido de datos, el fine-tuning supervisado de un modelo transformer deriva en sobreeajuste, por lo que los embeddings resultantes no reflejarían estructura semántica real.

Por ello, se utilizó el modelo base directamente como extractor de características, sin ningún tipo de ajuste supervisado. Por tanto, los embeddings extraídos a partir del token [CLS] capturaron la semántica general del texto tal y como la codificó el modelo preentrenado, sin estar sesgados hacia la tarea de clasificación de fake news. El resto del pipeline (reducción dimensional con UMAP y visualización) se mantuvo idéntico al análisis principal.

### 3.7. Validación estadística

Para validar la significancia estadística de la separación observada, se aplicó la **prueba U de Mann-Whitney**, un test no paramétrico que contrasta si dos distribuciones independientes son estadísticamente diferentes. Se calculó el $p$-valor para ambas dimensiones UMAP de forma independiente, aplicando corrección de Bonferroni para comparaciones múltiples. 

### 3.8. Implementación Agéntica

Dentro del sistema desarrollado se integró un agente de inteligencia artificial encargado de automatizar el análisis de las noticias extraídas y clasificarlas en función de la presencia o ausencia de clickbait. Para ello, se desarrolló una aplicación en Streamlit que combina técnicas de scraping web, lectura de feeds RSS y procesamiento de lenguaje natural mediante un modelo LLM de OpenAI, GPT-OSS-120b, integrado con LangChain.

En primer lugar, la aplicación obtiene las noticias de los distintos medios digitales españoles ya mencionados en la sección 3.1 rutilizando las mismas técincas de scraping respectivamente. Dependiendo del periódico, la extracción se realiza a partir de feeds RSS o mediante scraping directo del HTML. De cada noticia se recopilan campos como el enlace original, el periódico, la fecha, el título, el subtítulo, la categoría y el contenido del artículo.

Una vez extraídas, las noticias se incorporan a una cola de pendientes y son analizadas por el agente. El modelo evalúa principalmente la relación entre el titular y el contenido para detectar rasgos habituales del clickbait, como exageración, ambigüedad, suspense artificial, carga emocional excesiva o desajuste entre lo prometido por el titular y la información real del texto. Como resultado, cada noticia se clasifica como *Clickbait* o *No Clickbait*, acompañada de un score de confianza y una breve explicación del criterio utilizado.

Los resultados se almacenan en archivos JSON separados por medio, evitando duplicados y conservando el progreso de cada ejecución. Además, cuando una noticia es clasificada como clickbait, el agente propone un titular alternativo más neutro, informativo y fiel al contenido original. La aplicación también genera logs de seguimiento y mantiene una carpeta de noticias pendientes para conservar aquellas que no hayan podido procesarse correctamente y permitir su análisis posterior.

## 4. Resultados

### 4.1. Extracción de características

A partir de la distancia coseno entre el título y los párrafos del contenido, combinada con las diferencias lingüísticas descritas anteriormente, se entrenó un clasificador mediante regresión logística. Otros clasificadores más complejos mostraron sobreajuste, por lo que se optó por este modelo más simple.

<div id="Figura1" style="display:flex; gap:20px;">

<div style="flex:1; text-align:center;">

<img src="img/embeding_coseno_simple.png" width="75%">

<p><strong>Figura 1.</strong> Espacio embebido UMAP que relaciona el título con el cuerpo de la noticia.</p>

</div>

<div style="flex:1; text-align:center;">

<img src="img/ROC_caracteristicas.png" width="70%">

<p><strong>Figura 2.</strong> Curva ROC del clasificador basado en características lingüísticas.</p>

</div>

</div>


En la Figura 1, en la que el número encima de cada punto corresponde al párrafo del cuerpo, se observa que las noticias con mayor similitud coseno entre título y contenido tienden a agruparse en regiones próximas del espacio embebido. Sin embargo, como refleja la Figura 2, la capacidad discriminativa del clasificador resultante es limitada (AUC = 0.660), lo que evidencia que las características lingüísticas manuales no son suficientes para una detección robusta del clickbait. Se ha validado a partir de la clasificación de Taniwa, hacerlo con GPT sería una futura implementación.

### 4.2. Modelo Taniwa

#### 4.2.1. Embedding tras fine-tuning

Tras el fine-tuning con las etiquetas del sistema Taniwa, los embeddings resultantes muestran una separación clara entre ambas clases. La siguiente figura
revela clusters bien definidos de clickbait y no-clickbait, tanto para títulos como para contenido.

<div style="display:flex; gap:20px; align-items:flex-start;">

<div style="flex:1; text-align:center;">

<img src="img/UMAP_titulos_clickbait_taniwa.png" width="90%">

<p>
<strong>Figura 3.</strong>
Representación UMAP de los embeddings tras fine-tuning para títulos (Taniwa).
</p>

</div>

<div style="flex:1; text-align:center;">

<img src="img/UMAP_contenido_clickbait_taniwa.png" width="90%">

<p>
<strong>Figura 4.</strong>
Representación UMAP de los embeddings tras fine-tuning para contenido (Taniwa).
</p>

</div>

</div>

La separación visual observada fue validada mediante la prueba U de Mann-Whitney, que confirmó significancia estadística ($p \ll 0.05$) en ambas dimensiones UMAP. Cabe recordar que esta separación es en parte consecuencia del fine-tuning supervisado: el modelo optimiza sus representaciones para discriminar las etiquetas de entrenamiento, por lo que la separación en el espacio embebido no implica necesariamente que el clickbait sea intrínsecamente distinguible a nivel lingüístico.

#### 4.2.2. Variación por categoría temática
La Figura 5 presenta los espacios embebidos
segmentados por categoría temática, usando las etiquetas de Taniwa.

<div id="fig-umap-finetuned-categoria-taniwa" style="display:flex; gap:20px; align-items:flex-start;">

<div style="flex:1; text-align:center;">

<img src="img/UMAP_titulos_categoria_taniwa.png" width="100%">

<p>
<strong>Figura 5.</strong>
Representación UMAP de títulos segmentada por categoría temática (Taniwa).
</p>

</div>

<div style="flex:1; text-align:center;">

<img src="img/UMAP_contenido_categoria_taniwa.png" width="100%">

<p>
<strong>Figura 6.</strong>
Representación UMAP de contenido segmentada por categoría temática (Taniwa).
</p>

</div>

</div>

La categoría *Cultura* presenta una mayor propensión hacia el clickbait, evidenciada por su concentración en la región del espacio UMAP asociada a esta etiqueta. Las categorías *Internacional* y *Nacional* muestran una distribución más equilibrada entre ambas clases.

#### 4.2.3. Variación por medio de comunicación

La Figura 7 presenta la distribución segmentada por medio de comunicación. Los resultados no evidencian que ningún periódico concentre sus
contenidos de forma sistemática en una sola clase.

<div id="fig-umap-finetuned-periodico-taniwa" style="display:flex; gap:20px; align-items:flex-start;">

<div style="flex:1; text-align:center;">

<img src="img/UMAP_titulos_periodico_taniwa.png" width="100%">

<p>
<strong>Figura 7.</strong>
Representación UMAP de títulos segmentada por medio de comunicación (Taniwa).
</p>

</div>

<div style="flex:1; text-align:center;">

<img src="img/UMAP_contenido_periodico_taniwa.png" width="100%">

<p>
<strong>Figura 8.</strong>
Representación UMAP de contenido segmentada por medio de comunicación (Taniwa).
</p>

</div>

</div>

Los periódicos analizados presentan distribuciones heterogéneas en ambas regiones del espacio, sin que exista un medio que muestre una preferencia marcada hacia el clickbait.

### 4.3. Modelo GPT



#### 4.3.1. Embeddings tras fine-tuning con etiquetas del agente GPT

Se repitió el análisis utilizando las etiquetas generadas por el agente GPT, que clasifica
las noticias teniendo en cuenta tanto el título como el contenido. La
Figura 9 muestra los resultados obtenidos.

<div id="fig-umap-finetuned-gpt" style="display:flex; gap:20px; align-items:flex-start;">

<div style="flex:1; text-align:center;">

<img src="img/UMAP_titulos_clickbait_gpt.png" width="100%">

<p>
<strong>Figura 9.</strong>
Representación UMAP de los embeddings tras fine-tuning para títulos (GPT).
</p>

</div>

<div style="flex:1; text-align:center;">

<img src="img/UMAP_contenido_clickbait_gpt.png" width="100%">

<p>
<strong>Figura 10.</strong>
Representación UMAP de los embeddings tras fine-tuning para contenido (GPT).
</p>

</div>

</div>

Al igual que con las etiquetas de Taniwa, se observa una separación clara entre ambas
clases en el espacio UMAP, validada estadísticamente con la prueba U de Mann-Whitney
($p \ll 0.05$).

#### 4.3.2. Variación por categoría temática (GPT)

La Figura 11 muestra la distribución por categoría
temática con las etiquetas del agente GPT. Los patrones son coherentes con los observados
en el análisis con etiquetas Taniwa: la categoría *Cultura* concentra más contenidos
clasificados como clickbait, mientras que *Internacional* y *Nacional* presentan
distribuciones más equilibradas.

<div id="fig-umap-finetuned-categoria-gpt" style="display:flex; gap:20px; align-items:flex-start;">

<div style="flex:1; text-align:center;">

<img src="img/UMAP_titulos_categoria_gpt.png" width="100%">

<p>
<strong>Figura 11.</strong>
Representación UMAP de títulos segmentada por categoría temática (GPT).
</p>

</div>

<div style="flex:1; text-align:center;">

<img src="img/UMAP_contenido_categoria_gpt.png" width="100%">

<p>
<strong>Figura 12.</strong>
Representación UMAP de contenido segmentada por categoría temática (GPT).
</p>

</div>

</div>


#### 4.3.3. Variación por medio de comunicación (GPT)

La Figura 13 presenta la distribución por medio de
comunicación con etiquetas GPT. De forma consistente con el análisis anterior, no se
observa ningún periódico con una preferencia sistemática hacia el clickbait.

<div id="fig-umap-finetuned-periodico-gpt" style="display:flex; gap:20px; align-items:flex-start;">

<div style="flex:1; text-align:center;">

<img src="img/UMAP_titulos_periodico_gpt.png" width="100%">

<p>
<strong>Figura 13.</strong>
Representación UMAP de títulos segmentada por medio de comunicación (GPT).
</p>

</div>

<div style="flex:1; text-align:center;">

<img src="img/UMAP_contenido_periodico_gpt.png" width="100%">

<p>
<strong>Figura 14.</strong>
Representación UMAP de contenido segmentada por medio de comunicación (GPT).
</p>

</div>

</div> 

### 4.4. Modelo para vídeos de YouTube

#### 4.4.1. Embeddings con vídeos de YouTube etiquetados

Como parte del análisis adicional que se realizó sobre el conjunto de datos obtenidos de los vídeos de YouTube, se aplicó el mismo proceso de reducción UMAP sobre los embeddings extraídos, tanto para los títulos como para el contenido de los vídeos, coloreando los puntos según su etiqueta de Fake News o No Fake News.

<div id="fig-umap-finetuned-gpt" style="display:flex; gap:20px; align-items:flex-start;">

<div style="flex:1; text-align:center;">

<img src="img/UMAP_titulos_FakeNews.png" width="100%">

<p>
<strong>Figura 15.</strong>
Representación UMAP de los embeddings de los vídeos para títulos.
</p>

</div>

<div style="flex:1; text-align:center;">

<img src="img/UMAP_contenido_FakeNews.png" width="100%">

<p>
<strong>Figura 16.</strong>
Representación UMAP de los embeddings de los vídeos para contenido.
</p>

</div>

</div>

Los resultados obtenidos muestran, en ambos casos, una ausencia de separación clara entre ambas clases. En el espacio de embeddings de títulos, los puntos de ambas categorías aparecen distribuidos de forma entremezclada dentro de los mismos clústeres, sin que pueda identificarse ninguna frontera de separación. En el espacio de contenido la situación es aún más homogénea, con ambas clases completamente solapadas en el clúster principal y únicamente un outlier aislado que representa un vídeo con características atípicas respecto al resto del corpus.

Estos resultados son coherentes con las limitantes del análisis. Al no disponer de fine-tuning, el modelo no ha sido guiado para discriminar entre vídeos con feak news y no fake news, por lo que los embeddings no contienen señal supervisada que permita separar ambas clases. A esto se suma el reducido tamaño del corpus ($147$ vídeos), que hace que UMAP no tenga suficiente masa de datos para construir una geometría estable en el espacio reducido.

#### 4.4.2. Variación por categoría temática (YouTube)

Como parte del mismo análisis adicional sobre los vídeos de YouTube, se exploró la distribución de los embeddings segmentada por categoría temática (11s, Cambio climático y Covid), utilizando las etiquetas Fake News y No Fake News.

<div id="fig-umap-finetuned-categoria-gpt" style="display:flex; gap:20px; align-items:flex-start;">

<div style="flex:1; text-align:center;">

<img src="img/UMAP_titulos_categoria_FakeNews.png" width="100%">

<p>
<strong>Figura 17.</strong>
Representación UMAP de los títulos de los vídeos segmentada por categoría temática.
</p>

</div>

<div style="flex:1; text-align:center;">

<img src="img/UMAP_contenido_categoria_FakeNews.png" width="100%">

<p>
<strong>Figura 18.</strong>
Representación UMAP del contenido de los vídeos segmentada por categoría temática.
</p>

</div>

</div>

En este caso los resultados presentan una estructura un poco más interpretable que la obtenida con la separación por etiqueta. En el espacio de títulos se aprecian dos regiones diferenciadas dentro del espacio UMAP, en el que los vídeos de Covid tienden a agruparse en la zona superior-derecha del espacio, mientras que los vídeos de 11s y Cambio climático coexisten en la región inferior-izquierda, aunque sin una frontera entre ellos. Esta separación parcial es esperable teniendo en cuenta que los títulos de vídeos sobre Covid incorporan terminología médica que el modelo codifica en regiones distintas del espacio semántico, mientras que los vídeos sobre el 11s y el cambio climático comparten un lenguaje más genérico de denuncia y conspiración que los aproxima en el espacio embebido.

En el espacio de contenido, la separación por categoría es prácticamente inexistente, con las tres temáticas completamente mezcladas en el clúster principal. Esto sugiere que las descripciones o transcripciones de los vídeos presentan estilos más homogéneos que los títulos, convergiendo hacia un registro común independientemente del tema tratado.
En cualquier caso, los resultados de separación por categoría, aunque ligeramente más informativos que los de separación por etiqueta, tampoco alcanzan la claridad observada en el análisis principal. 

### 4.5. Comparación entre títulos y contenido

En ambos sistemas de etiquetado (Taniwa y GPT), el modelo entrenado sobre títulos mostró una convergencia más estable durante el entrenamiento y una discriminación más robusta en la comparación posterior. Este resultado es coherente con la naturaleza del clickbait como fenómeno del titular, el título es el elemento diseñado para generar curiosidad o engañar, mientras que el cuerpo de la noticia tiende a ser más informativo y neutral.

El modelo de contenido presenta mayor inestabilidad en las curvas de entrenamiento, lo que puede atribuirse a la longitud de los textos (hasta 512 tokens), al tamaño reducido del corpus y al truncado que descarta información de las noticias más largas.

Los resultados obtenidos en el análisis adicional de vídeos de YouTube contrastan notablemente con los del análisis principal. Mientras que en el corpus de noticias el fine-tuning permitió obtener embeddings con una separación estadísticamente significativa entre clickbait y no-clickbait tanto en títulos como en contenido, el análisis de YouTube arroja representaciones sin estructura discriminativa clara en ninguna de las dos dimensiones textuales.

### 4.6. Comparación entre clasificadores: BERT vs. Taniwa

Para evaluar en qué medida el modelo BERT (entrenado con etiquetas GPT) reproduce las decisiones del sistema Taniwa, se realizó una inferencia sobre el corpus de
Taniwa y se compararon las clasificaciones mediante el coeficiente Kappa de Cohen, que mide el acuerdo entre dos clasificadores corrigiendo la fracción de coincidencias atribuibles al azar. Es importante recalcar que las noticias de Taniwa y GPT son distintas, por lo que se va hacer inferencia con un conjunto de noticias que GPT nunca ha visto.

Con el objetivo de evitar el sobreajuste durante el entrenamiento con etiquetas GPT, se utilizaron únicamente 4 épocas tanto para el modelo de título como para el de contenido, junto con un *learning rate* de $3 \cdot 10^{-5}$. Las curvas de entrenamiento confirmaron una convergencia estable y sin sobreajuste. Para el título, el error de entrenamiento descendió de 0.68 a 0.62 y el de validación de 0.68 a 0.63; para el contenido, el error de entrenamiento pasó de 0.69 a 0.64 y el de validación de 0.69 a 0.62.

<div id="fig-umap-finetuned-gpt" style="display:flex; gap:20px; align-items:flex-start;">

<div style="flex:1; text-align:center;">

<img src="img/agente_acuerdo_titulo.png" width="100%">

<p>
<strong>Figura 19.</strong>
Comparación entre BERT y Taniwa para títulos.
</p>

</div>

<div style="flex:1; text-align:center;">

<img src="img/agente_acuerdo_contenido.png" width="100%">

<p>
<strong>Figura 20.</strong>
Comparación entre BERT y Taniwa para contenido.
</p>

</div>

</div>

Los resultados se resumen en la siguiente tabla.

<div id="tab-kappa" style="display:flex; justify-content:center; margin:20px 0;">

<table>
<thead>
<tr>
<th>Métrica</th>
<th>Títulos</th>
<th>Contenido</th>
</tr>
</thead>

<tbody>
<tr>
<td><strong>Acuerdo simple</strong></td>
<td>82.0%</td>
<td>62.6%</td>
</tr>

<tr>
<td><strong>Cohen's Kappa</strong></td>
<td>0.405</td>
<td>0.023</td>
</tr>

<tr>
<td><strong>Interpretación</strong></td>
<td>Moderado</td>
<td>Bajo</td>
</tr>
</tbody>
</table>

</div>

<p align="center">
<strong>Tabla 1.</strong>
Acuerdo entre BERT (entrenado con GPT) y Taniwa.
</p>

Las representaciones de la figura anterior permiten analizar visualmente la distribución espacial de los acuerdos y desacuerdos. En el caso de los títulos, se aprecia una zona de mayor densidad (verde) donde ambos modelos coinciden en la detección de clickbait, lo que lleva a pensar que existe una señal semántica común en los titulares. En cambio, para el contenido no se observan distribuciones espaciales claras, lo que indica que la coincidencia entre modelos es menos clara.

Esta asimetría entre título y contenido puede explicarse por las diferencias en los datos de entrenamiento de cada clasificador: Taniwa fue entrenado únicamente sobre títulos, por lo que sus representaciones capturan señales propias de los titulares. El modelo BERT, entrenado con las etiquetas de GPT que considera tanto el título como el cuerpo, incorpora información adicional del contenido que Taniwa no ha procesado. Por lo tanto, el criterio entre ambos es más efectiva cuando se evalúa el título, y empeora al analizar el cuerpo de la noticia.


En cuanto a las discrepancias por medio de comunicación:

| Periódico            | Título | Contenido |
| -------------------- | -----: | --------: |
| El Confidencial      |  **35.0%** |     **50.0%** |
| RTVE                 |  25.0% |     43.3% |
| ABC                  |  18.3% |     28.3% |
| okdiario             |  16.7% |     38.3% |
| La Vanguardia        |  15.0% |     25.0% |
| ElDiario             |  15.0% |     35.0% |
| 20minutos            |  15.0% |     30.0% |
| Huffingtonpost       |  13.3% |     45.0% |
| Mediterráneo Digital |   8.3% |     41.7% |



 El Confidencial concentra el mayor porcentaje de desacuerdos tanto en título como en contenido. Otro resultado interesante es que ambos modelos discrepan poco (observando el título) con el Mediterráneo Digital y Huffingtonpost que son los periódicos que estaban buscados con mayor clickbait.



Por categoría, la discrepacia es la siguiente:


| Categoría     | Título | Contenido |
| ------------- | -----: | --------: |
| Cultura       |  **33.8%** |     43.8% |
| Internacional |  12.1% |     **44.7%** |
| Nacional      |  10.5% |     24.7% |


Estos patrones son coherentes con las diferencias de criterio entre ambos sistemas. Taniwa tiende a clasificar como clickbait las noticias de Cultura por sus títulos llamativos, mientras que GPT, al revisar el contenido, no las identifica como clickbait. Por el contrario, GPT clasifica una proporción mayor de noticias de Internacional (aunque muy cerca está cultura) como clickbait al analizar el cuerpo, lo que Taniwa no puede detectar al trabajar solamente con el titular. Estas discrepancias reflejan, por tanto, no solo las limitaciones de cada modelo, sino también la diferencia conceptual entre ambas aproximaciones al clickbait.

Un Kappa de $0.41$ indica un acuerdo moderado, lo que refleja que ambos clasificadores comparten el criterio de detección de forma parcial pero no completa. Esta divergencia es esperable: GPT razona de forma holística sobre el texto completo, mientras que BERT aprende patrones estadísticos locales a partir de un corpus limitado. El acuerdo moderado obtenido en títulos sugiere que ambos sistemas capturan señales similares en los titulares, mientras que el acuerdo bajo en contenido evidencia que el cuerpo de la noticia no contiene una señal de clickbait igualmente accesible para ambos enfoques.


Por útlimo, se han seleccionado algunos ejemplos representativos de casos de desacuerdo entre BERT-GPT y Taniwa. Solo se van a comprar los títulos ya que estos han sido los únicos capaces de separar.

**Ejemplo 1: Titular sensacionalista que solo Taniwa identifica**
- Título: "Rosalía canta malamente y explica por qué ya no interpreta el mal querer tras leer la pancarta crítica de un fan"
- Periódico: 20minutos
- Categoría: Cultura
- Taniwa: Clickbait | BERT: No Clickbait (score: 0.456)

En este caso Taniwa lo detectó como Clickbait mientras que Bert-GPT no. Se observa que el título tiene palabras sensacionalistas como "malamente" que Taniwa ha podido identificar como clickbait. 

**Ejemplo 2: Noticia que solo BERT detecta clickbait**
- Título: "Amine Kessaci: la narcocracia crece en los lugares que el Estado ha dejado abandonados"
- Periódico: ABC 
- Categoría: Internacional
- Taniwa: No Clickbait | BERT: Clickbait (score: 0.551)

A pesar del tono informativo, la colocación de un personaje como protagonista del titular y el uso de estructuras más emotivas es un patrón que BERT asocia con clickbait basándose en su entrenamiento con GPT.

**Ejemplo 3: Noticia claramente clasificada como No Clickbait**
- Título: "Tiroteo en el centro de Atenas: un hombre de 89 años deja cinco heridos y es detenido tras darse a la fuga"
- Periódico: 20minutos 
- Categoría: Internacional
- Ambos: No Clickbait (score BERT: 0.223)

Titular directo, informativo y sin elementos sensacionalistas.

**Ejemplo 4: Noticia claramente identificada como Clickbait**
- Título: "El Rey Carlos III trolea al presidente Trump en un brindis en la Casa Blanca: si no fuera por nosotros ustedes hablarían francés"
- Periódico: 20minutos 
- Categoría: Internacional
- Ambos: Clickbait (score BERT: 0.616)

Uso explícito de palabras emotivas ("trolea"). Ambos modelos reconocen este patrón consistentemente.



## 5. Conclusiones



Analizando los resultados, se observa que la detección de características para detectar clickbait no ha sido exitosa. Tras probar diferentes métricas y hacer muchas pruebas, las dos únicas variables más destacables para el clasificador han sido la similitud del coseno y la diferencia lingüistica, ambas entre el título y el cuerpo. Obteniendo como resultado un AUC=0.660. Esta comparación solo se hace con las etiquetas de Taniwa, un análisis futuro podría ser comparar también con GPT.

Cambiando la perspectiva, se decidió hacer un fine-tunning en el modelo de BERT con el objetivo de destilar los modelos de GPT y Taniwa. Para este caso, se consiguieron separar los datos, aunque no es comparable con la extracción de características ya que esto se hace de manera supervisada (en base a las etiquetas clasificadas previamente). Los resultados obtenidos: el modelo de Taniwa detecta como clickbait las noticias de Cultura, independientemente del medio. Lo interesante es que ha conseguido separar tanto el cuerpo como el título de una manera clara a pesar de que Taniwa solo clasificaba por títulos. Entre nacional y cultura resulta más difícil separar ambas distribuciones. Para el periódico no se encontraron distribuciones separadas.

Los resultados obtenidos del modelo GPT cuya definición de clickbait es comparar el cuerpo con el título es diferente al de Taniwa. Se observa que las categorías con menor densidad de clickbait es Cultura, mientras que la que más tiene es Internacional, este resultado se puede deber a que, aunque los titulares de Cultura suelen ser más emotivos, el título y el contenido no se desvían mucho. Con los medios de comunicación se obtienen los mismos resultados.

Con el objetivo de comparar ambos modelos, se entrenó el modelo de GPT con el embeding de BERT y haciendo inferencia con los datos de Taniwa para ver qué noticias clasificaba como clickbait y no clickbait. Los resultados obtenidos muestran que para el título sí es capaz de coincidir en las noticias clickbait, concentrando estos valores de coincidencia en una región aislada del UMAP. Sin embargo, para el cuerpo no es capaz de identificarlo bien, posiblemente debido a que Taniwa solo clasifica por el título.




Un futuro análisis podría ser hacer el fine-tunning por medios de comunicación, de esta manera se podría ver qué medios de comunicación tienen diferentes formas semánticas y poder hacer inferencia. 





## 6. Bibliografía

1. Bazaco, A., Redondo, M., y Sánchez-García, P. (2019). El clickbait como estrategia del periodismo viral: concepto y metodología. Revista Latina de Comunicación Social, (74), 94-115.

2. Bravo Araujo, A., Serrano-Puche, J., y Novoa Jaso, M. (2021). Uso del clickbait en los medios nativos digitales españoles. Un análisis de El Confidencial, El Español, Eldiario.es y Ok Diario. Doxa Comunicación. Revista Interdisciplinar de Estudios de Comunicación y Ciencias Sociales, (32), 185-210.

3. Taniwa (2023). taniwasl/clickbait_es. Hugging Face. Recuperado de https://huggingface.co/taniwasl/clickbait_es

4. Devlin, J., Chang, M.-W., Lee, K., y Toutanova, K. (2018).  
   *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*.  
   CoRR, abs/1810.04805.  
   Disponible en: http://arxiv.org/abs/1810.04805

5. Mann, H. B., y Whitney, D. R. (1947).  
   *On a Test of Whether one of Two Random Variables is Stochastically Larger than the Other*.  
   The Annals of Mathematical Statistics, 18(1), 50–60.  
   https://doi.org/10.1214/aoms/1177730491

6. jopebrui. (2026). fake-news-detector [Vídeos de YouTube con títulos, transcripción y puntuación de Fake New]. GitHub. https://github.com/jopebrui/fake-news-detector

7. Cohen, J. (1960). A coefficient of agreement for nominal scales. *Educational and Psychological Measurement*, 20(1), 37–46. https://doi.org/10.1177/001316446002000104

8. Landis, J. R., & Koch, G. G. (1977). The measurement of observer agreement for categorical data. *Biometrics*, 33(1), 159–174. https://doi.org/10.2307/2529310

- *Información sobre los periódicos analizados*
